In [1]:
import duckdb

In [6]:
con = duckdb.connect()
con.sql("""
CREATE SECRET r2 (
  TYPE r2,
  KEY_ID 'd7531cdc5a5bdf2b8393fa8b78af38ea',
  SECRET '89452704d72b9049c3fbb8198777be7bd44589aa27082b4d560729964a3bcb70',
  ACCOUNT_ID '308fb6c6165381d81a5073fdb3c211b4'
)""")

SuccessBOOLEAN
True


In [7]:
con.sql("""
CREATE VIEW ev AS
SELECT * FROM read_parquet(
  'r2://wiki-lake/raw/wiki_recentchange/source=kowiki/*/*/*.parquet',
  hive_partitioning = true)
QUALIFY row_number() OVER (PARTITION BY event_id ORDER BY ingested_at) = 1  -- 중복 제거
""")

CountBIGINT


In [8]:
# 전체 규모, 종류별 건수
con.sql("SELECT op, count(*) FROM ev GROUP BY op").show()

┌──────────────────┬──────────────┐
│        op        │ count_star() │
│     varchar      │    int64     │
├──────────────────┼──────────────┤
│ log_delete_redir │            1 │
│ delete           │            1 │
│ create           │           89 │
│ update           │         6738 │
└──────────────────┴──────────────┘



In [9]:
# 본문 조회가 잘 됐는지
con.sql("SELECT content_status, count(*) FROM ev GROUP BY 1").show()


┌────────────────┬──────────────┐
│ content_status │ count_star() │
│    varchar     │    int64     │
├────────────────┼──────────────┤
│ none           │            2 │
│ ok             │         6827 │
└────────────────┴──────────────┘



In [10]:
# 많이 고쳐진 문서 Top 10
con.sql("SELECT page_title, count(*) AS edits FROM ev GROUP BY 1 ORDER BY 2 DESC LIMIT 10").show()


┌───────────────────────────────┬───────┐
│          page_title           │ edits │
│            varchar            │ int64 │
├───────────────────────────────┼───────┤
│ 2026년 한국시리즈             │    28 │
│ 2026년 K리그2의 경기 결과     │    21 │
│ 야호(夜好)라디오              │    11 │
│ 루스 존슨 (미시간주의 정치인) │    11 │
│ MARETU                        │    11 │
│ 딕 포스투무스                 │    11 │
│ KBO 한국시리즈                │    10 │
│ 암살자(들)                    │    10 │
│ 동구 (부산광역시)             │    10 │
│ 핑크 ( MARETU )               │    10 │
└───────────────────────────────┴───────┘
  10 rows                     2 columns



In [11]:
# 되돌림 태그가 붙은 편집
con.sql("""
SELECT page_title, rev_id, tags, event_dt FROM ev
WHERE list_has_any(tags, ['mw-rollback','mw-undo','mw-manual-revert'])
ORDER BY event_dt DESC LIMIT 10
""").show()

┌───────────────────────┬──────────┬────────────────────────────────────────────────────────────────────────────────────────┬──────────────────────────┐
│      page_title       │  rev_id  │                                          tags                                          │         event_dt         │
│        varchar        │  int64   │                                       varchar[]                                        │         varchar          │
├───────────────────────┼──────────┼────────────────────────────────────────────────────────────────────────────────────────┼──────────────────────────┤
│ 피구세                │ 42774214 │ [visualeditor, mw-manual-revert]                                                       │ 2026-11-07T14:15:47.027Z │
│ 앵무새                │ 42773770 │ [mw-manual-revert]                                                                     │ 2026-11-07T13:48:54.665Z │
│ 만세력                │ 42772907 │ [mw-undo, wikieditor]                                 

In [12]:
# 실제 본문 하나 보기 (앞 500자)
con.sql("SELECT page_title, left(content, 500) FROM ev WHERE content IS NOT NULL LIMIT 1").show(max_width=200)

┌──────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│  page_title  │                                                                                "left"("content", 500)                                                                                 │
│   varchar    │                                                                                        varchar                                                                                        │
├──────────────┼───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ 중앙정보국장 │ [[파일:Flag of the U.S. Director of Central Intelligence.png|섬네일]]\n[[파일:Cia-lobby-seal.jpg|섬네일|오른쪽|본부 건물 로비에 있는 직경 {{단위 변환|16|ft|m|0|adj=on}}의 화강암 CIA │
│              │  인장.]]\n[[파일:CIA

In [ ]:
# 봇 편집 , 사람 편집 cnt

con.sql("SELECT is_bot, count(*) AS n, round(100.0*count(*)/sum(count(*)) OVER (),1) AS pct FROM ev GROUP BY 1").show()

┌─────────┬───────┬────────┐
│ is_bot  │   n   │  pct   │
│ boolean │ int64 │ double │
├─────────┼───────┼────────┤
│ false   │  1236 │   18.1 │
│ true    │  5593 │   81.9 │
└─────────┴───────┴────────┘



In [14]:
con.sql("SELECT comment, count(*) FROM ev WHERE is_bot GROUP BY 1 ORDER BY 2 DESC LIMIT 5").show(max_width=150)

┌────────────────────────────────────────────────────────────────────┬──────────────┐
│                              comment                               │ count_star() │
│                              varchar                               │    int64     │
├────────────────────────────────────────────────────────────────────┼──────────────┤
│ 봇: 전거 통제 틀 추가                                              │         3809 │
│ 봇: 공백 정리                                                      │         1246 │
│ 0 개의 출처 구조, 1 개의 링크를 깨진 것으로 표시 #IABot (v2.0.9.7) │          205 │
│ 1 개의 출처 구조, 0 개의 링크를 깨진 것으로 표시 #IABot (v2.0.9.7) │           60 │
│ 0 개의 출처 구조, 2 개의 링크를 깨진 것으로 표시 #IABot (v2.0.9.7) │           55 │
└────────────────────────────────────────────────────────────────────┴──────────────┘

